# VoxMind Backend — Colab Runner

Run the cells below **in order, top to bottom**, on a GPU runtime
(Runtime -> Change runtime type -> GPU, e.g. T4).

1. Clone the repo and install Python dependencies.
2. Install and start Ollama, then pull the LLM (`llama3.2:3b`).
3. Download the Piper TTS voice files.
4. Install `cloudflared` (public tunnel client).
5. Start the VoxMind backend (FastAPI + WebSocket) in the background.
6. Open a Cloudflare Tunnel to the backend and print its public URL.

**After Cell 6 prints a `https://<something>.trycloudflare.com` URL**, take
that host and use it in the frontend's connection field as:

```
wss://<that-host>/ws
```

Note: the Colab session is ephemeral. Every time you restart the runtime
or re-run Cell 6, you get a **new** tunnel URL and must re-paste it into
the frontend.
**v2 note:** starting the backend (Cell 5) also downloads the
speech-emotion model (~380 MB, one-time per session) for Auto mood detection.
Set `DISABLE_SER=1` before starting the backend to skip it.


In [ ]:
# Cell 1 — clone repo & install
!git clone https://github.com/abbinavv/VoxMind.git voxmind && cd voxmind && pip install -q -r backend/requirements.txt

In [ ]:
# Cell 1b - verify the Piper API resolved correctly (fail fast, before starting the backend)
import importlib
piper = importlib.import_module('piper')
from piper import PiperVoice, SynthesisConfig  # must exist in piper-tts >= 1.4 (piper1-gpl)
assert hasattr(PiperVoice, 'load'), 'Wrong piper package: PiperVoice.load missing'
print('piper-tts OK:', getattr(piper, '__version__', 'unknown'))

In [ ]:
# Cell 2 - install & start Ollama, pull model
# Colab's image lacks zstd, which the Ollama installer needs to unpack. Install it first.
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
!ollama pull llama3.2:3b


In [ ]:
# Cell 3 - download the Piper voice (Amy: warm, natural female) into /content
!wget -q -O /content/en_US-amy-medium.onnx https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/amy/medium/en_US-amy-medium.onnx
!wget -q -O /content/en_US-amy-medium.onnx.json https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/amy/medium/en_US-amy-medium.onnx.json
print('voice downloaded:', __import__('os').path.getsize('/content/en_US-amy-medium.onnx'), 'bytes')


In [ ]:
# Cell 4 — install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

In [ ]:
# Cell 5 - start backend in background
# Absolute path to the Piper voice (downloaded to /content in Cell 3).
# Stream logs to a file so a startup crash is visible instead of silently detached.
import os, subprocess, time
os.environ['PIPER_MODEL'] = '/content/en_US-amy-medium.onnx'
backend_log = open('/content/backend.log', 'w')
proc = subprocess.Popen(
    ['python', '-m', 'colab.run_backend'],
    cwd='voxmind',
    stdout=backend_log, stderr=subprocess.STDOUT,
    env={**os.environ},
)
time.sleep(15)  # give the models time to load
log = open('/content/backend.log').read()
if proc.poll() is not None:
    print('BACKEND EXITED EARLY - full log below:')
    print(log)
else:
    print('Backend running (pid', proc.pid, '). Recent log:')
    print(log[-2000:])


In [ ]:
# UPDATE CELL - pull the latest code and restart the backend (no full re-setup)
# Run this after new fixes are merged to main, to update a running Colab session.
import os, subprocess, time
subprocess.run(['git', 'pull'], cwd='voxmind')
# make sure Ollama is alive
if not subprocess.run(['pgrep','-f','ollama'],capture_output=True,text=True).stdout.strip():
    subprocess.Popen(['ollama','serve']); time.sleep(6)
# restart the backend with the updated code
subprocess.run(['pkill','-f','colab.run_backend']); time.sleep(3)
os.environ['PIPER_MODEL'] = '/content/en_US-amy-medium.onnx'
backend_log = open('/content/backend.log','w')
proc = subprocess.Popen(['python','-m','colab.run_backend'], cwd='voxmind',
                        stdout=backend_log, stderr=subprocess.STDOUT, env={**os.environ})
time.sleep(15)
print('updated + restarted, pid', proc.pid)
print(open('/content/backend.log').read()[-1200:])
# NOTE: the tunnel (Cell 6) keeps its URL - no need to re-run it or re-paste the URL.


In [ ]:
# Cell 6 — open tunnel and print URL (use wss://<printed-host>/ws in the frontend)
!./cloudflared tunnel --url http://localhost:8000